In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np

In [ ]:
FRECUENCIA_MUESTREO = 4096
DURACION = 8
FRECUENCIA_SENO = 123
AMPLITUD_SENO = 0.1
SEMILLA = 0

rng = np.random.default_rng(SEMILLA)

In [ ]:
def dft(x):
    n = len(x)
    m = np.arange(n)
    giros = np.exp(-2j * np.pi * np.outer(m, m) / n)
    return giros @ x

In [ ]:
def fft(x):
    n = len(x)
    if n == 1:
        return np.array(x, dtype=complex)
    if n % 2 != 0:
        raise ValueError("la longitud tiene que ser una potencia de 2")
    pares = fft(x[0::2])
    impares = fft(x[1::2]) * np.exp(-2j * np.pi * np.arange(n // 2) / n)
    return np.concatenate([pares + impares, pares - impares])

In [ ]:
prueba = rng.normal(size=1024)
print(np.abs(dft(prueba) - np.fft.fft(prueba)).max())
print(np.abs(fft(prueba) - np.fft.fft(prueba)).max())

In [ ]:
def medir(funcion, x):
    inicio = time.perf_counter()
    funcion(x)
    return time.perf_counter() - inicio


longitudes = 2 ** np.arange(4, 12)
tiempos_dft = []
tiempos_fft = []
tiempos_numpy = []
for n in longitudes:
    x = rng.normal(size=n)
    tiempos_dft.append(medir(dft, x))
    tiempos_fft.append(medir(fft, x))
    tiempos_numpy.append(medir(np.fft.fft, x))

In [ ]:
plt.loglog(longitudes, tiempos_dft, "o-", label="dft")
plt.loglog(longitudes, tiempos_fft, "o-", label="fft")
plt.loglog(longitudes, tiempos_numpy, "o-", label="numpy.fft")
plt.xlabel("Muestras")
plt.ylabel("Tiempo (s)")
plt.legend();

In [ ]:
n = FRECUENCIA_MUESTREO * DURACION
t = np.arange(n) / FRECUENCIA_MUESTREO
seno = AMPLITUD_SENO * np.sin(2 * np.pi * FRECUENCIA_SENO * t)
datos = seno + rng.normal(size=n)

In [ ]:
plt.plot(t, datos, label="seno + ruido")
plt.plot(t, seno, label="seno")
plt.xlim(0, 0.1)
plt.xlabel("Tiempo (s)")
plt.legend();

In [ ]:
espectro = fft(datos)
print(np.abs(espectro - np.fft.fft(datos)).max())

frecuencias = np.arange(n // 2) / DURACION
potencia = np.abs(espectro[: n // 2]) ** 2
print(f"Pico en {frecuencias[np.argmax(potencia)]} Hz, {potencia.max() / np.median(potencia):.0f} veces la mediana")

In [ ]:
plt.semilogy(frecuencias, potencia)
plt.xlim(0, 500)
plt.xlabel("Frecuencia (Hz)")
plt.ylabel("|X(f)|²");